# Experiment 19 — V3 LLM Annotation of 10 Struggling Students
# For comparison against human annotations (Pranay + Arundhati)

## Purpose
Run the V3/CCPP prompt (no mental model) on all 10 struggling students' 
non-perfect submissions. Save LLM gap tags per student in the same format 
as human annotation JSONs for direct κ comparison.

## Paths (root = /mnt/d/Projects/kintsugi/)
- Student code inputs: `scripts/annotation_tool/annotation_inputs/student_{sid}.json`
- Problem KC mapping: `dataset/CodeWorkout/Problem_Prompts/problem_prompts.csv`
- V3 prompt builder: `lib/v3_prompt.py` → `build_v3_prompt()`
- Output (per student): `results/human_validation/llm_v3_10students/llm_v3_annotations_{sid}.json`

# Experiment 19 — V3 Annotation of 10 Struggling Students

Run V3/CCPP prompt on all 10 human-annotated struggling students.
No mental model injection — V3 Baseline only.
Save after each student completes.

In [ ]:
import json
import os
import sys
import time
import pandas as pd
from pathlib import Path
from datetime import datetime
from google import genai
from google.genai import types

# Add project root to path
ROOT = Path("/mnt/d/Projects/kintsugi")
sys.path.insert(0, str(ROOT))

from lib.v3_prompt import build_v3_prompt
from utils.constants import GEMINI_API_KEY

API_KEY = GEMINI_API_KEY or os.environ.get("GOOGLE_API_KEY")

# --- Configuration ---
MODEL_ID = "gemini-2.5-flash"
SLEEP_SECONDS = 7  # Free tier: 7 sec. Paid tier: 3 sec.
TEMPERATURE = 0.3

STUDENT_IDS = [10155, 9948, 14189, 14352, 14362, 14363, 14374, 14414, 14474, 14499]

INPUT_DIR = ROOT / "scripts" / "annotation_tool" / "annotation_inputs"
OUTPUT_DIR = ROOT / "results" / "human_validation" / "llm_v3_10students"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load problem_prompts.csv
pp_df = pd.read_csv(ROOT / "dataset" / "CodeWorkout" / "Problem_Prompts" / "problem_prompts.csv")

KC_COLUMNS = [
    "If/Else", "NestedIf", "While", "For", "NestedFor",
    "Math+-*/", "Math%", "LogicAndNotOr", "LogicCompareNum", "LogicBoolean",
    "StringFormat", "StringConcat", "StringIndex", "StringLen",
    "StringEqual", "CharEqual", "ArrayIndex", "DefFunction"
]

VALID_KCS = set(KC_COLUMNS)

def get_required_kcs(problem_id):
    row = pp_df[pp_df['ProblemID'] == problem_id]
    if row.empty:
        return []
    row = row.iloc[0]
    return [kc for kc in KC_COLUMNS if pd.notna(row.get(kc)) and float(row.get(kc)) == 1.0]

def get_problem_info(problem_id):
    row = pp_df[pp_df['ProblemID'] == problem_id]
    if row.empty:
        return None, None
    row = row.iloc[0]
    return row['Requirement'], int(row['AssignmentID'])

print(f"Project root: {ROOT}")
print(f"Model: {MODEL_ID}")
print(f"Students: {STUDENT_IDS}")
print(f"Problems in curriculum: {len(pp_df)}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# --- Gemini API setup (google-genai SDK) ---
if not API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running Experiment 19.")

client = genai.Client(api_key=API_KEY)

def call_gemini(prompt_text):
    """Send prompt to Gemini and return raw text response."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt_text,
        config=types.GenerateContentConfig(
            temperature=TEMPERATURE,
        ),
    )
    return response.text or ""

print("Gemini client configured.")

In [ ]:
# --- Load all student data ---
student_data = {}

for sid in STUDENT_IDS:
    input_path = INPUT_DIR / f"student_{sid}.json"
    with open(input_path, 'r') as f:
        data = json.load(f)
    student_data[sid] = data
    
    submissions = data['submissions']
    n_total = len(submissions)
    n_perfect = sum(1 for v in submissions.values() if v['score'] >= 1.0)
    n_nonperfect = n_total - n_perfect
    print(f"Student {sid}: {n_total} problems ({n_perfect} perfect, {n_nonperfect} to annotate)")

total_calls = sum(
    sum(1 for v in d['submissions'].values() if v['score'] < 1.0)
    for d in student_data.values()
)
est_minutes = (total_calls * SLEEP_SECONDS) / 60
print(f"\nTotal API calls needed: {total_calls}")
print(f"Estimated runtime at {SLEEP_SECONDS}s sleep: {est_minutes:.0f} minutes")

In [ ]:
# --- Main run loop ---
# Processes one student at a time. Saves JSON after each student.
# If a student's output file already exists, SKIPS that student (checkpoint).

def parse_llm_response(raw_text):
    """Parse the JSON response from Gemini. Returns (parsed_dict, status)."""
    cleaned = (raw_text or "").strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    
    try:
        parsed = json.loads(cleaned)
        return parsed, "ok"
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and start < end:
            try:
                parsed = json.loads(cleaned[start:end + 1])
                return parsed, "ok_extracted_json"
            except json.JSONDecodeError as e:
                return {"reasoning": raw_text, "knowledge_gaps": []}, f"parse_error: {e}"
        return {"reasoning": raw_text, "knowledge_gaps": []}, "parse_error: no_json_object"


def clean_gaps(parsed):
    """Return only valid KC names from the parsed LLM response."""
    gaps = parsed.get("knowledge_gaps", [])
    if isinstance(gaps, str):
        gaps = [gaps]
    if not isinstance(gaps, list):
        return []
    return [gap for gap in gaps if gap in VALID_KCS]


all_results = {}

for student_idx, sid in enumerate(STUDENT_IDS):
    output_path = OUTPUT_DIR / f"llm_v3_annotations_{sid}.json"
    
    # --- Checkpoint: skip if already done ---
    if output_path.exists():
        print(f"\n[{student_idx+1}/10] Student {sid} — ALREADY DONE, skipping.")
        with open(output_path, 'r') as f:
            all_results[sid] = json.load(f)
        continue
    
    print(f"\n{'='*60}")
    print(f"[{student_idx+1}/10] Student {sid}")
    print(f"{'='*60}")
    
    submissions = student_data[sid]['submissions']
    annotations = {}
    call_count = 0
    raw_responses = {}
    errors = []
    
    sorted_pids = sorted(submissions.keys(), key=lambda x: int(x))
    
    for pid_str in sorted_pids:
        pid = int(pid_str)
        sub = submissions[pid_str]
        code = sub['code']
        score = sub['score']
        
        # --- Skip perfect submissions ---
        if score >= 1.0:
            annotations[pid_str] = {"gaps": []}
            continue
        
        requirement, assignment_id = get_problem_info(pid)
        required_kcs = get_required_kcs(pid)
        
        if requirement is None:
            print(f"  WARNING: Problem {pid} not found in problem_prompts.csv, skipping.")
            annotations[pid_str] = {"gaps": []}
            raw_responses[pid_str] = {"error": "problem_not_found"}
            continue
        
        prompt = build_v3_prompt(
            problem_id=pid,
            requirement=requirement,
            assignment_id=assignment_id,
            required_kcs=required_kcs,
            student_code=code,
            score=score
        )
        
        call_count += 1
        try:
            start_time = time.time()
            raw_response = call_gemini(prompt)
            elapsed = time.time() - start_time
            
            parsed, status = parse_llm_response(raw_response)
            gaps = clean_gaps(parsed)
            
            annotations[pid_str] = {"gaps": gaps}
            raw_responses[pid_str] = {
                "raw_response": raw_response,
                "parsed_response": parsed,
                "parse_status": status,
                "time_sec": round(elapsed, 3),
                "score": score,
                "assignment_id": assignment_id,
                "required_kcs": required_kcs,
                "invalid_kcs": [gap for gap in parsed.get("knowledge_gaps", []) if gap not in VALID_KCS] if isinstance(parsed.get("knowledge_gaps", []), list) else []
            }
            
            gap_str = ", ".join(gaps) if gaps else "(none)"
            print(f"  P{pid} (score={score:.2f}) → [{gap_str}] ({elapsed:.1f}s)")
            
        except Exception as e:
            error_msg = f"API error on P{pid}: {str(e)}"
            print(f"  ERROR: {error_msg}")
            errors.append(error_msg)
            annotations[pid_str] = {"gaps": []}
            raw_responses[pid_str] = {
                "error": str(e),
                "score": score,
                "assignment_id": assignment_id,
                "required_kcs": required_kcs
            }
        
        time.sleep(SLEEP_SECONDS)
    
    result = {
        "rater": "LLM_Gemini_V3",
        "studentId": str(sid),
        "student_id": str(sid),
        "model_id": MODEL_ID,
        "temperature": TEMPERATURE,
        "exportDate": datetime.now().isoformat(),
        "total_problems": len(submissions),
        "totalAnnotated": len(annotations),
        "total_calls": call_count,
        "errors": errors,
        "annotations": annotations,
        "raw_responses": raw_responses
    }
    
    with open(output_path, 'w') as f:
        json.dump(result, f, indent=2)
    
    all_results[sid] = result
    
    n_with_gaps = sum(1 for a in annotations.values() if a.get("gaps"))
    print(f"\n  SAVED: {output_path.name}")
    print(f"  Calls: {call_count} | With gaps: {n_with_gaps} | Errors: {len(errors)}")

print(f"\n{'='*60}")
print("ALL STUDENTS COMPLETE")
print(f"{'='*60}")

In [ ]:
# --- Summary ---
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"\n{'StudentID':>10} {'Total':>6} {'Calls':>6} {'WithGaps':>9} {'Errors':>7}")
print("-" * 45)

total_calls_all = 0
total_gaps_all = 0

for sid in STUDENT_IDS:
    if sid not in all_results:
        print(f"{sid:>10}  — NOT RUN")
        continue
    
    r = all_results[sid]
    ann = r['annotations']
    n_total = r.get('total_problems', len(ann))
    n_calls = r.get('total_calls', len(r.get('raw_responses', {})))
    n_with_gaps = sum(1 for a in ann.values() if a.get("gaps"))
    n_errors = len(r.get('errors', []))
    
    total_calls_all += n_calls
    total_gaps_all += n_with_gaps
    
    print(f"{sid:>10} {n_total:>6} {n_calls:>6} {n_with_gaps:>9} {n_errors:>7}")

print("-" * 45)
print(f"{'TOTAL':>10} {'':>6} {total_calls_all:>6} {total_gaps_all:>9}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")

In [ ]:
# --- Quick sanity check: compare against human annotations ---
HUMAN_DIR = ROOT / "dataset" / "Rater_KC_Tags" / "Rated_KC_V3"

pranay_files = list(HUMAN_DIR.glob("kc_annotations_Pranay Ghuge_10155_*.json"))
if pranay_files:
    with open(pranay_files[0], 'r') as f:
        human_ann = json.load(f)
    
    llm_ann = all_results.get(10155, {}).get('annotations', {})
    
    print("Sanity check — Student 10155, first 5 non-perfect problems:")
    print(f"\n{'ProblemID':>10} {'Human Gaps':<40} {'LLM Gaps':<40}")
    print("-" * 90)
    
    count = 0
    for pid_str in sorted(human_ann.get('annotations', {}).keys(), key=lambda x: int(x)):
        h_gaps = human_ann['annotations'][pid_str].get('gaps', [])
        l_gaps = llm_ann.get(pid_str, {}).get('gaps', [])
        
        if not h_gaps and not l_gaps:
            continue
        
        h_str = ", ".join(h_gaps) if h_gaps else "(none)"
        l_str = ", ".join(l_gaps) if l_gaps else "(none)"
        print(f"{pid_str:>10} {h_str:<40} {l_str:<40}")
        
        count += 1
        if count >= 5:
            break
else:
    print("No human annotation found for student 10155 — skipping sanity check.")

print("\nDone. Ready for κ analysis.")